# Inference
## Text Completion

In [2]:
import torch
from transformers import GPTNeoForCausalLM, GPT2Tokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model_id = "EleutherAI/gpt-neo-2.7B"
tokenizer = GPT2Tokenizer.from_pretrained(model_id)
model = GPTNeoForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16).to(device)

Loading weights:   0%|          | 0/420 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: EleutherAI/gpt-neo-2.7B
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
transformer.h.{0...30}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0...31}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
prompt = "The story so far: in the beginning, the universe was created."
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

In [4]:
print(device)
print(next(model.parameters()).device)

mps
mps:0


In [6]:
generated_ids = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.9,
    max_length=200,
    pad_token_id=50256
)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)

The story so far: in the beginning, the universe was created. God created
the whole wide, wonderful, complex world in seven days, then rested on
the seventh day, having nothing more to do on this earth. And God was
happy.

Of course, from this point on, everything changed. Life started going
downhill in an unending cycle of destruction and ruin. Soon this cycle
would take us out into space, and then back to earth.

And then it was time for creation to end, and God to take a break. But
not the kind of break we have today. God wanted to go on forever.
So, in six days, God decided to make a huge jump, and create the first
star.

The next few days God spent creating the first planet. It was a
planet called Earth. And God, as you know, said, "You were only being
creative."

Then God


In [7]:
prompts = [
    "Once there was a man ",
    "The weather today will be ",
    "A great soccer player must "
]

tokenizer.padding_side = "left"
tokenizer.pad_token = tokenizer.eos_token
encoding = tokenizer(prompts, padding=True, return_tensors='pt').to(device)
with torch.no_grad():
    generated_ids = model.generate(
        **encoding,
        do_sample=True,
        temperature=0.9,
        max_length=50,
        pad_token_id=50256
    )
generated_texts = tokenizer.batch_decode(
    generated_ids, skip_special_tokens=True
)


In [8]:
for text in generated_texts:
    print('---------')
    print(text)

---------
Once there was a man 
Who liked to build things.

#  **PATTERNONG**

#  _from_ **WALLACE ALLEN**   
_a novel_

_Wall
---------
The weather today will be  hot with a high near 87. There is a  20% chance of rain showers this afternoon. Chance of precipitation is 20%.

Detailed Sunday afternoon and overnight weather:

High: 80F

---------
A great soccer player must  
Learn to love every ball that  
He kicks with his foot,  
For there's only one thing  
That keeps him in this World.  
And that is  


## Few-Shot Learning

In [9]:
prompt = """
Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: neutral
"""
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

generated_ids = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.9,
    max_length=200,
    pad_token_id=50256
)
generated_text = tokenizer.decode(generated_ids[0])
print(generated_text)


Sentence: This movie is very nice.
Sentiment: positive

#####

Sentence: I hated this movie, it sucks.
Sentiment: negative

#####

Sentence: This movie was actually pretty funny.
Sentiment: positive

#####

Sentence: This movie could have been better.
Sentiment: neutral

#####

Sentence: This movie was great.
Sentiment: positive

#####

Sentence: It was a really funny movie.
Sentiment: positive

#####

Sentence: This movie was actually the best movie I have ever seen.
Sentiment: positive

#####

Sentence: The movie was great. It made me think.
Sentiment: positive

#####

Sentence: This movie was a really bad movie, I couldn't take it.
Sentiment: positive

####


## Code Generation

In [10]:
prompt = """Instruction: Generate a Python function that lets you reverse a list of integers.

Answer: """
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)

In [11]:
generated_ids = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.9,
    max_length=200,
    pad_token_id=50256
)
generated_text = tokenizer.decode(generated_ids[0])

## Evaluating the Generated Content
Using `lm-evaluation-harness`

**Install**
```bash
git clone https://github.com/EleutherAI/lm-evaluation-harness
cd lm-evaluation-harness
pip install -e .
```

**Execute**
```bash
lm_eval --model hf \
      --model_args pretrained=EleutherAI/gpt-neo-2.7B,dtype=float16 \
      --tasks wikitext \
      --device mps
```

**Output**
```
hf ({'pretrained': 'EleutherAI/gpt-neo-2.7B', 'dtype': 'float16'}), gen_kwargs: ({}), limit: None, num_fewshot: None, batch_size: 1
| Tasks  |Version|Filter|n-shot|    Metric     |   | Value |   |Stderr|
|--------|------:|------|-----:|---------------|---|------:|---|------|
|wikitext|      2|none  |     0|bits_per_byte  |↓  | 0.7108|±  |   N/A|
|        |       |none  |     0|byte_perplexity|↓  | 1.6368|±  |   N/A|
|        |       |none  |     0|word_perplexity|↓  |13.9402|±  |   N/A|
```